# The LLM study — open-weight models with Ollama on Google Colab

**Decisions D35 and D43.** The study compares self-hosted, open-weight models only (D43):
no evidence is sent to a model vendor's API. The models run here, in a Colab GPU runtime, because UTS
iHPC is decommissioned and Cetus never became reachable. Colab was suggested by the
supervisor on 15/09/2026.

**This notebook is a launcher, not pipeline code.** Every result is produced by
`scripts/run_llm_study.py` from `configs/llm.yaml`, the same as on the Mac. Nothing below
changes what a report says; it only provides a GPU, an Ollama server and somewhere safe to
write.

### Before running
1. **Runtime → Change runtime type → GPU.** Without a GPU, Ollama still runs, on CPU,
   extremely slowly, and without complaint. Cell 2 refuses to continue.
2. Keep this tab in the foreground. Idle sessions are reclaimed first.

### Why the whole run happens inside this notebook
Colab's free tier disallows serving web services from the runtime, so Ollama's port is
**not** tunnelled back to the Mac. The model is called over `localhost`, which is also the
honest model of an on-premises deployment: the evidence never leaves the machine that
hosts the model.

### If the session disconnects
The VM disk is wiped, but reports are written to **Google Drive** one at a time. Run
every cell again from the top: the models re-download, and the study resumes at the first
detection with no report. A resume onto a different GPU is recorded and flagged; a resume
onto different model weights is refused.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/rowanjames04/GNN-LLM-NIDS-Capstone.git"
# Pin a commit for a real run, so the code that produced the reports is fixed.
# Every report records the commit anyway; pinning makes a resume use the same one.
REF = "main"

PROMPT_VERSIONS = ["v1"]      # ["v1", "v2"] for the full study; "v1" alone for the pilot
N_REPORTS = None              # None = the config's n_reports; a small number for a pilot

# Reports go to Drive so a disconnect loses nothing. False writes to the VM disk,
# which is wiped when the session ends -- only sensible for a quick pilot.
USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/capstone-llm-study/reports"

## 2. Confirm a GPU is attached

In [ ]:
import shutil, subprocess

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then run again.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 3. Code and a minimal install

In [ ]:
import os

if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
subprocess.run(["git", "-C", "/content/repo", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", "/content/repo", "checkout", "-q", REF], check=True)
os.chdir("/content/repo")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

# requirements-llm.txt, not requirements.txt: the LLM path needs no torch.
subprocess.run(["pip", "install", "-q", "-r", "requirements-llm.txt"], check=True)

## 4. Install and start Ollama

`zstd` is needed by the installer to unpack the release; `pciutils` and `lshw` let it
detect the GPU (without them it installs CPU-only).

In [ ]:
import time, urllib.request

subprocess.run("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw",
               shell=True, check=True)
if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

server = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2) as r:
            print("Ollama server up:", r.read().decode())
            break
    except OSError:
        time.sleep(1)
else:
    raise SystemExit("Ollama did not start -- see /content/ollama.log")

## 5. Pull the self-hosted models named in the roster

The models come from `configs/llm.yaml` (entries with `billing: self_hosted`), not from
this notebook, so the Mac and Colab cannot disagree about what was run. Tags move when a
model is re-published: **quote the digest printed below, not the tag.**

In [ ]:
import yaml

cfg = yaml.safe_load(open("configs/llm.yaml"))
models = [e["model"] for e in cfg["roster"]
          if e.get("enabled", True) and e.get("billing") == "self_hosted"]
print("self-hosted roster:", models)
for m in models:
    subprocess.run(["ollama", "pull", m], check=True)
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 6. Point the output at Drive

Writes a copy of the config with only `output.dir` changed. Everything else -- packs,
prompts, model options, `num_ctx` -- is the committed configuration.

In [ ]:
out_dir = DRIVE_DIR
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    out_dir = "/content/reports"
os.makedirs(out_dir, exist_ok=True)

cfg["output"]["dir"] = out_dir
with open("/content/llm_colab.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("reports ->", out_dir)

## 7. Run the self-hosted arm

`--only self_hosted` runs the Ollama entries, which are the whole study apart from the
template control (run on the Mac, in seconds). No spend flag is needed, because nothing
here is billed.

**Safe to re-run.** Completed reports are skipped. Exit status 75 means the run stopped
for an infrastructure reason (e.g. the Ollama server died) with everything so far saved --
restart cell 4, then re-run this cell.

In [ ]:
cmd = ["python", "scripts/run_llm_study.py", "--config", "/content/llm_colab.yaml",
       "--only", "self_hosted", "--prompt-versions", *PROMPT_VERSIONS]
if N_REPORTS:
    cmd += ["-n", str(N_REPORTS)]
print(" ".join(cmd), "\n")
subprocess.run(cmd)

## 8. Take the results home

Completed files are named `reports_ollama_<model>_<version>.json`. A `.partial.jsonl` file
means that model has not finished -- re-run cell 7.

On the Mac: copy the completed `.json` files into `results/reports/`, run
`python scripts/summarise_llm_study.py`, and commit. The Mac repo stays the source of
truth; Drive only holds working copies.

In [ ]:
import glob

for path in sorted(glob.glob(f"{out_dir}/*")):
    print(f"{os.path.getsize(path):>10,}  {os.path.basename(path)}")

if not USE_DRIVE:
    from google.colab import files
    for path in glob.glob(f"{out_dir}/reports_ollama_*.json"):
        files.download(path)